# Proyecto Final

In [106]:
# ===== IMPORTAR LIBRERÍAS DE PYSPARK =====
from pyspark.sql import SparkSession  # Para crear y manejar sesiones de Spark
from pyspark.sql import functions as F  # Funciones de transformación de DataFrames
from pyspark.sql import types as T  # Tipos de datos de Spark
from pyspark.sql import Window  # Para funciones de ventana (window functions)

# ===== CREAR SESIÓN DE SPARK =====
# Inicializar la sesión de Spark con un nombre de aplicación
spark = SparkSession.builder \
    .appName("Pract2_PySpark") \
    .getOrCreate()

# Verificar la versión de Spark instalada
print("Versión de Spark:", spark.version)

# Obtener el contexto de Spark (SparkContext) para operaciones de bajo nivel
sc = spark.sparkContext

# ===== DEFINIR RUTA DE DATOS =====
# Ruta donde están almacenados los archivos CSV y Parquet
DATA_PATH = "/home/FinPlus/work/data/"


Versión de Spark: 3.5.0


## Impotamos la data

In [62]:
# ===== CARGAR DATOS DE CLIENTES =====
# Leer archivo CSV con información de clientes
# - header='true': la primera fila contiene nombres de columnas
# - delimiter=',': las columnas se separan por comas
client = (spark.read.option('header', 'true').option('delimiter',',')
                     .csv(DATA_PATH + 'CLIENTS.csv'))

# Mostrar los primeros 5 registros (cliente.show(5) sin truncar columnas)
# 0 significa mostrar todas las columnas sin límite de ancho
client.show(5, 0)


+------------+----------------------+-----------------+------+------------+--------------+-----------+--------------+--------------+--------------------+------------------+------------------+-------------+--------------+-----------+-----------------+-------+-----------+------------------+------------------+------------------+---------------------+------------------+----------+--------------+----------+--------------------------+--------+---------------------+------------------------+------------------------+------------------------+---------------------------+---------------------------+---------------------------+-----------------------+-----------------------+-----------------------+----------------------+----------------------+-------------------+---------------------+-----------------+-------------------+----------------+
|CLIENT_ID   |NON_COMPLIANT_CONTRACT|NAME_PRODUCT_TYPE|GENDER|TOTAL_INCOME|AMOUNT_PRODUCT|INSTALLMENT|EDUCATION     |MARITAL_STATUS|HOME_SITUATION      |REGION_SC

In [ ]:
# ===== VER ESQUEMA DEL DATAFRAME CLIENT =====
# Muestra el tipo de dato de cada columna
# y si puede contener valores nulos
client.printSchema()


In [ ]:
# ===== LISTAR NOMBRES DE LAS COLUMNAS =====
# Devuelve una lista con todos los nombres de las columnas de Clients
client.columns


In [63]:
# ===== CARGAR DATOS DE BEHAVIOURAL =====
# Leer datos en formato Parquet
beh = (spark.read.parquet(DATA_PATH + 'BEHAVIOURAL_PARQUET'))

# Renombrar columna 'CURRENCY' a 'B_CURRENCY' para evitar conflictos al unir con client
beh = beh.withColumnRenamed('CURRENCY', 'B_CURRENCY')

# Mostrar los primeros 5 registros sin truncar columnas
beh.show(5, 0)


+------------------+------------+----------+--------------------+-----------------+------------------------+--------------------+------------------------+--------------------------+-------------------+-------------------+---------------+------------------+----------+
|CONTRACT_ID       |CLIENT_ID   |DATE      |CREDICT_CARD_BALANCE|CREDIT_CARD_LIMIT|CREDIT_CARD_DRAWINGS_ATM|CREDIT_CARD_DRAWINGS|CREDIT_CARD_DRAWINGS_POS|CREDIT_CARD_DRAWINGS_OTHER|CREDIT_CARD_PAYMENT|NUMBER_DRAWINGS_ATM|NUMBER_DRAWINGS|NUMBER_INSTALMENTS|B_CURRENCY|
+------------------+------------+----------+--------------------+-----------------+------------------------+--------------------+------------------------+--------------------------+-------------------+-------------------+---------------+------------------+----------+
|ES1821190439i00XXX|ES182363269V|2020-08-22|4189.73             |5400.0           |1193.4                  |1193.4              |0.0                     |0.0                       |162.0          

In [ ]:
# ===== VER ESQUEMA DEL DATAFRAME BEHAVIOURAL =====
# Mostrar tipos de datos y compatibilidad con nulos
beh.printSchema()


In [ ]:
# ===== LISTAR COLUMNAS DE BEHAVIOURAL =====
# Obtener lista de todos los nombres de columnas
beh.columns


# Quitamos los duplicados en los dos data sets

In [ ]:
# ===== ELIMINAR DUPLICADOS EN CLIENT POR CLIENT_ID =====
# dropDuplicates(['CLIENT_ID']): mantiene solo la primera ocurrencia de cada CLIENT_ID
# Esto elimina registros duplicados basándose en el ID del cliente
client_sin_duplicados_por_id = client.dropDuplicates(['CLIENT_ID'])

# Mostrar los primeros 5 registros sin duplicados
client_sin_duplicados_por_id.show(5)

# Comparar el número de filas antes y después
print(f"Filas originales en client: {client.count()}")
print(f"Filas después de eliminar duplicados (por CLIENT_ID) en client: {client_sin_duplicados_por_id.count()}")


+------------+----------------------+-----------------+------+------------+--------------+-----------+---------+--------------+----------------+------------+------------------+-------------+--------------+-----------+-----------------+-------+-----------+------------------+------------------+------------------+---------------------+------------------+----------+--------------+----------+--------------------------+--------+---------------------+------------------------+------------------------+------------------------+---------------------------+---------------------------+---------------------------+-----------------------+-----------------------+-----------------------+----------------------+----------------------+-------------------+---------------------+-----------------+-------------------+----------------+
|   CLIENT_ID|NON_COMPLIANT_CONTRACT|NAME_PRODUCT_TYPE|GENDER|TOTAL_INCOME|AMOUNT_PRODUCT|INSTALLMENT|EDUCATION|MARITAL_STATUS|  HOME_SITUATION|REGION_SCORE|      AGE_IN_YEARS|J

In [ ]:
# ===== ELIMINAR DUPLICADOS EN BEHAVIOURAL =====
# Elimina filas duplicadas basándose en múltiples columnas:
# CLIENT_ID, DATE (fecha), CREDICT_CARD_BALANCE (saldo), CONTRACT_ID (ID contrato)
# Solo mantiene la primera ocurrencia de cada combinación única
beh_sin_duplicados_por_id = beh.dropDuplicates(['CLIENT_ID', 'DATE', 'CREDICT_CARD_BALANCE', 'CONTRACT_ID'])

# Mostrar primeros 5 registros sin duplicados
beh_sin_duplicados_por_id.show(5)

# Comparar número de filas antes y después
print(f"Filas originales en beh: {beh.count()}")
print(f"Filas después de eliminar duplicados (por CLIENT_ID) en beh: {beh_sin_duplicados_por_id.count()}")


+------------------+------------+----------+--------------------+-----------------+------------------------+--------------------+------------------------+--------------------------+-------------------+-------------------+---------------+------------------+--------+
|       CONTRACT_ID|   CLIENT_ID|      DATE|CREDICT_CARD_BALANCE|CREDIT_CARD_LIMIT|CREDIT_CARD_DRAWINGS_ATM|CREDIT_CARD_DRAWINGS|CREDIT_CARD_DRAWINGS_POS|CREDIT_CARD_DRAWINGS_OTHER|CREDIT_CARD_PAYMENT|NUMBER_DRAWINGS_ATM|NUMBER_DRAWINGS|NUMBER_INSTALMENTS|CURRENCY|
+------------------+------------+----------+--------------------+-----------------+------------------------+--------------------+------------------------+--------------------------+-------------------+-------------------+---------------+------------------+--------+
|ES1821489396v00XXX|ES182100006A|2021-07-29|                 0.0|           3240.0|                     0.0|                 0.0|                     0.0|                       0.0|                0.0| 

# Contamos el número de nulls que hay en cada una de las columnas

In [ ]:
# ===== CONTAR VALORES NULOS EN CADA COLUMNA DE CLIENT =====
# Itera sobre cada columna y cuenta cuántos valores nulos (NULL) contiene
for column in client.columns:
    # Filtra filas donde la columna es NULL y cuenta el resultado
    null_count = client.filter(client[column].isNull()).count()
    print(f"Columna '{column}': {null_count} valores nulos")


In [ ]:
# ===== CONTAR VALORES NULOS EN CADA COLUMNA DE BEHAVIOURAL =====
# Lo mismo que para client: contar nulos en cada columna
for column in beh.columns:
    null_count = beh.filter(beh[column].isNull()).count()
    print(f"Columna '{column}': {null_count} valores nulos")


# Cambiamos Nulls

In [64]:
# ===== PREPARAR DATAFRAME CLIENT: CONVERTIR TIPOS Y LLENAR NULOS =====

# Copiar client a df_client para trabajar sin modificar el original
df_client = client

# ===== PASO 1: CONVERTIR SCORING COLUMNS A FLOAT =====
# Las columnas de scoring deben ser numéricas (float) para cálculos
score_columns = ['REACTIVE_SCORING', 'PROACTIVE_SCORING', 'BEHAVIORAL_SCORING']

for col_name in score_columns:
    # Convertir cada columna de scoring al tipo float
    df_client = df_client.withColumn(col_name, F.col(col_name).cast(T.FloatType()))

# ===== PASO 2: CALCULAR PROMEDIOS PARA ELIMINAR Y CMABIAR NULOS =====
# Calcular el promedio de cada scoring (excluyendo nulos)
# Estos promedios se usarán para reemplazar valores nulos
avg_reactive_scoring = df_client.filter(F.col('REACTIVE_SCORING').isNotNull()).agg(F.avg('REACTIVE_SCORING')).collect()[0][0]
avg_proactive_scoring = df_client.filter(F.col('PROACTIVE_SCORING').isNotNull()).agg(F.avg('PROACTIVE_SCORING')).collect()[0][0]
avg_behavioral_scoring = df_client.filter(F.col('BEHAVIORAL_SCORING').isNotNull()).agg(F.avg('BEHAVIORAL_SCORING')).collect()[0][0]

# ===== PASO 3: DEFINIR ESTRATEGIA DE IMPUTACIÓN DE NULOS =====
# Diccionario que especifica qué valor usar para reemplazar nulos en cada columna
fill_values = {
    'INSTALLMENT': 0.0,  # Si cuota de instalación es nula, usar 0
    'EDUCATION': 'Bachelor',  # Si educación es nula, asumir Bachelor
    'MARITAL_STATUS': 'NA',  # Si estado civil es nulo, usar 'NA'
    'JOB_SENIORITY': 0.0,  # Si antigüedad laboral es nula, usar 0
    'CAR_AGE': 0.0,  # Si edad del coche es nula, usar 0
    'FAMILY_SIZE': 1.0,  # Si tamaño familia es nulo, asumir 1 persona
    'REACTIVE_SCORING': avg_reactive_scoring,  # Usar promedio
    'PROACTIVE_SCORING': avg_proactive_scoring,
    'BEHAVIORAL_SCORING': avg_behavioral_scoring,
    'DAYS_LAST_INFO_CHANGE': 0.0,  # Si últimos cambios es nulo, usar 0
    'NUMBER_OF_PRODUCTS': 0.0,  # Si número de productos es nulo, usar 0
    'EMPLOYER_ORGANIZATION_TYPE': 'NA',  # Si tipo de organización es nulo, usar 'NA'
    'NUM_PREVIOUS_LOAN_APP': 0.0,  # Si solicitudes previas es nulo, usar 0
    'LOAN_ANNUITY_PAYMENT_MAX': 0.0,
    'LOAN_ANNUITY_PAYMENT_MIN': 0.0,
    'LOAN_ANNUITY_PAYMENT_SUM': 0.0,
    'LOAN_APPLICATION_AMOUNT_MAX': 0.0,
    'LOAN_APPLICATION_AMOUNT_MIN': 0.0,
    'LOAN_APPLICATION_AMOUNT_SUM': 0.0,
    'LOAN_CREDIT_GRANTED_MAX': 0.0,
    'LOAN_CREDIT_GRANTED_MIN': 0.0,
    'LOAN_CREDIT_GRANTED_SUM': 0.0,
    'LOAN_VARIABLE_RATE_MAX': 0.0,
    'LOAN_VARIABLE_RATE_MIN': 0.0,
    'NUM_STATUS_ANNULLED': 0.0,  
    'NUM_STATUS_AUTHORIZED': 0.0,  
    'NUM_STATUS_DENIED': 0.0,  
    'NUM_STATUS_NOT_USED': 0.0, 
    'NUM_FLAG_INSURED': 0.0  
}

# ===== PASO 4: APLICAR CAMBIOS =====
# Reemplazar todos los nulos según fill_values
df_client = df_client.na.fill(fill_values)

# ===== PASO 5: VERIFICAR QUE NO QUEDAN NULOS =====
print("Client DataFrame after handling missing values. Verifying null counts:")
for column in df_client.columns:
    null_count = df_client.filter(F.col(column).isNull()).count()
    if null_count > 0:
        print(f"Columna '{column}': {null_count} valores nulos")
    else:
        print(f"Columna '{column}': 0 valores nulos")

# Mostrar primeros 5 registros del dataframe limpio
df_client.show(5)


Client DataFrame after handling missing values. Verifying null counts:
Columna 'CLIENT_ID': 0 valores nulos
Columna 'NON_COMPLIANT_CONTRACT': 0 valores nulos
Columna 'NAME_PRODUCT_TYPE': 0 valores nulos
Columna 'GENDER': 0 valores nulos
Columna 'TOTAL_INCOME': 0 valores nulos
Columna 'AMOUNT_PRODUCT': 0 valores nulos
Columna 'INSTALLMENT': 0 valores nulos
Columna 'EDUCATION': 0 valores nulos
Columna 'MARITAL_STATUS': 0 valores nulos
Columna 'HOME_SITUATION': 0 valores nulos
Columna 'REGION_SCORE': 0 valores nulos
Columna 'AGE_IN_YEARS': 0 valores nulos
Columna 'JOB_SENIORITY': 0 valores nulos
Columna 'HOME_SENIORITY': 0 valores nulos
Columna 'LAST_UPDATE': 0 valores nulos
Columna 'OWN_INSURANCE_CAR': 0 valores nulos
Columna 'CAR_AGE': 0 valores nulos
Columna 'FAMILY_SIZE': 0 valores nulos
Columna 'REACTIVE_SCORING': 0 valores nulos
Columna 'PROACTIVE_SCORING': 0 valores nulos
Columna 'BEHAVIORAL_SCORING': 0 valores nulos
Columna 'DAYS_LAST_INFO_CHANGE': 0 valores nulos
Columna 'NUMBER_

# Union de datasets y cambio de tipo de datos

In [65]:
# ===== UNIR CLIENT Y BEHAVIOURAL DATASETS =====

# ===== PASO 1: OBTENER LA ÚLTIMA TRANSACCIÓN POR CLIENTE =====
# Crear una ventana ordenada por fecha descendent (más reciente primero)
ventana = Window.partitionBy("CLIENT_ID").orderBy(F.col("DATE").desc())

# Añadir un ranking (1 = más reciente, 2 = segunda más reciente, etc.)
df_beh_ranking = beh.withColumn("rank", F.row_number().over(ventana))

# Filtrar solo la fila con rank=1 (la más reciente) y eliminar la columna rank
df_best = df_beh_ranking.filter(F.col("rank") == 1).drop("rank")

# ===== PASO 2: UNIR CLIENT CON LA ÚLTIMA TRANSACCIÓN DE BEHAVIOURAL =====
# Realizar un LEFT JOIN: todos los clientes mas su última transacción (si existe)
# on="CLIENT_ID": unir por ID del cliente
# how="left": mantener todos los clientes aunque no tengan transacciones
df_clean = df_client.join(df_best, on="CLIENT_ID", how="left")

# Mostrar resultado de la unión
df_clean.show()


+------------+----------------------+-----------------+------+------------+--------------+-----------+---------+--------------+--------------------+------------------+------------------+-------------+--------------+-----------+-----------------+-------+-----------+----------------+-----------------+------------------+---------------------+------------------+----------+--------------+----------+--------------------------+--------+---------------------+------------------------+------------------------+------------------------+---------------------------+---------------------------+---------------------------+-----------------------+-----------------------+-----------------------+----------------------+----------------------+-------------------+---------------------+-----------------+-------------------+----------------+------------------+----------+--------------------+-----------------+------------------------+--------------------+------------------------+--------------------------+----

In [66]:
# ===== LLENAR NULOS DESPUÉS DE LA UNIÓN =====
# Después de unir, algunas columnas nuevas pueden tener valores nulos
# Definir valores por defecto para los nulos en las columnas de BEHAVIOURAL
nulos = {
    'CREDICT_CARD_BALANCE': 0.0,
    'CONTRACT_ID': 'NA',
    'DATE': 'NA',  
    'CREDIT_CARD_LIMIT': 0.0,
    'CREDIT_CARD_DRAWINGS_ATM': 0.0, 
    'CREDIT_CARD_DRAWINGS': 0.0, 
    'CREDIT_CARD_DRAWINGS_OTHER': 0.0, 
    'CREDIT_CARD_DRAWINGS_POS': 0.0,  
    'CREDIT_CARD_PAYMENT': 0.0, 
    'NUMBER_DRAWINGS_ATM': 0.0, 
    'NUMBER_DRAWINGS': 0.0,
    'NUMBER_INSTALMENTS': 0.0,  
    'B_CURRENCY': 'euros'
}

# Aplicar el cambio
df_clean = df_clean.na.fill(nulos)

# Mostrar resultado después de llenar nulos
df_clean.show()


+------------+----------------------+-----------------+------+------------+--------------+-----------+---------+--------------+--------------------+------------------+------------------+-------------+--------------+-----------+-----------------+-------+-----------+----------------+-----------------+------------------+---------------------+------------------+----------+--------------+----------+--------------------------+--------+---------------------+------------------------+------------------------+------------------------+---------------------------+---------------------------+---------------------------+-----------------------+-----------------------+-----------------------+----------------------+----------------------+-------------------+---------------------+-----------------+-------------------+----------------+------------------+----------+--------------------+-----------------+------------------------+--------------------+------------------------+--------------------------+----

In [67]:
# ===== CONVERTIR COLUMNAS A TIPO FLOAT (NUMÉRICO) =====
# Lista de columnas que deben ser numéricas para análisis posteriores
NewDataTypes = ['NON_COMPLIANT_CONTRACT',
 'TOTAL_INCOME',
 'AMOUNT_PRODUCT',
 'INSTALLMENT',
 'REGION_SCORE',
 'AGE_IN_YEARS',
 'JOB_SENIORITY',
 'HOME_SENIORITY',
 'LAST_UPDATE',
 'CAR_AGE',
 'FAMILY_SIZE',
 'REACTIVE_SCORING',
 'PROACTIVE_SCORING',
 'BEHAVIORAL_SCORING',
 'DAYS_LAST_INFO_CHANGE',
 'NUMBER_OF_PRODUCTS',
 'DIGITAL_CLIENT',
 'EMPLOYER_ORGANIZATION_TYPE',
 'NUM_PREVIOUS_LOAN_APP',
 'LOAN_ANNUITY_PAYMENT_MAX',
 'LOAN_ANNUITY_PAYMENT_MIN',
 'LOAN_ANNUITY_PAYMENT_SUM',
 'LOAN_APPLICATION_AMOUNT_MAX',
 'LOAN_APPLICATION_AMOUNT_MIN',
 'LOAN_APPLICATION_AMOUNT_SUM',
 'LOAN_CREDIT_GRANTED_MAX',
 'LOAN_CREDIT_GRANTED_MIN',
 'LOAN_CREDIT_GRANTED_SUM',
 'LOAN_VARIABLE_RATE_MAX',
 'LOAN_VARIABLE_RATE_MIN',
 'NUM_STATUS_ANNULLED',
 'NUM_STATUS_AUTHORIZED',
 'NUM_STATUS_DENIED',
 'NUM_STATUS_NOT_USED',
 'NUM_FLAG_INSURED',
 'CREDICT_CARD_BALANCE',
 'CREDIT_CARD_LIMIT',
 'CREDIT_CARD_DRAWINGS_ATM',
 'CREDIT_CARD_DRAWINGS',
 'CREDIT_CARD_DRAWINGS_POS',
 'CREDIT_CARD_DRAWINGS_OTHER',
 'CREDIT_CARD_PAYMENT',
 'NUMBER_DRAWINGS_ATM',
 'NUMBER_DRAWINGS',
 'NUMBER_INSTALMENTS']

# Iterar y convertir cada columna a float
for i in NewDataTypes:
    df_clean = df_clean.withColumn(i, F.col(i).cast('float'))

# Mostrar el esquema final (tipos de datos)
df_clean.printSchema()


root
 |-- CLIENT_ID: string (nullable = true)
 |-- NON_COMPLIANT_CONTRACT: float (nullable = true)
 |-- NAME_PRODUCT_TYPE: string (nullable = true)
 |-- GENDER: string (nullable = true)
 |-- TOTAL_INCOME: float (nullable = true)
 |-- AMOUNT_PRODUCT: float (nullable = true)
 |-- INSTALLMENT: float (nullable = true)
 |-- EDUCATION: string (nullable = false)
 |-- MARITAL_STATUS: string (nullable = false)
 |-- HOME_SITUATION: string (nullable = true)
 |-- REGION_SCORE: float (nullable = true)
 |-- AGE_IN_YEARS: float (nullable = true)
 |-- JOB_SENIORITY: float (nullable = true)
 |-- HOME_SENIORITY: float (nullable = true)
 |-- LAST_UPDATE: float (nullable = true)
 |-- OWN_INSURANCE_CAR: string (nullable = true)
 |-- CAR_AGE: float (nullable = true)
 |-- FAMILY_SIZE: float (nullable = true)
 |-- REACTIVE_SCORING: float (nullable = false)
 |-- PROACTIVE_SCORING: float (nullable = false)
 |-- BEHAVIORAL_SCORING: float (nullable = false)
 |-- DAYS_LAST_INFO_CHANGE: float (nullable = true)
 |--

# Transformacion de la data

# Actividad del cliente

VEMOS LA CANTIDAD DE PERSONAS QUE HACE MUCHO TIEMPO QUE NO ACTUALIZAN LOS DATOS Y PODRIAN ESTAR INACTIVOS AUN TENIENDO SALDO

In [ ]:
# ===== ANALIZAR ACTIVIDAD DEL CLIENTE =====
# Ver cuándo fue el último cambio de información del cliente
# DAYS_LAST_INFO_CHANGE: número de días desde la última actualización
(df_clean.groupBy('DAYS_LAST_INFO_CHANGE')  # Agrupar por días desde último cambio
    .count()  # Contar clientes en cada período
    .orderBy(F.desc('count'))  # Ordenar por cantidad (descendente)
).show(10,0)


ANALIZAMOS CUANTOS CLIENTES UTILIZAN EFECTIVO Y CUANTOS DIGITAL, Y CLASIFICAMOS SEGUN SU PORCENTAGE DE USO EN CASHLESS, EFECTIVO-DEPENDIENTE O MIXTO. (EL EFECTIVO ES CARO DE GESTIONAR)

In [73]:
# ===== SEGMENTACIÓN DE CLIENTES POR EFECTIVO VS DIGITAL =====

# ===== ANALIZAR CLIENTES DIGITALES =====
# Contar cuántos clientes son DIGITAL_CLIENT = '1' vs otros valores
(df_clean.groupBy(
    'DIGITAL_CLIENT').count().orderBy(F.desc('count'))
).show(10)

# ===== CALCULAR PORCENTAJE DE CLIENTES DIGITALES =====
# Contar clientes con DIGITAL_CLIENT='1' y dividir por el total
# Multiplicar por 100 para obtener porcentaje
Clientes_digitales = (df_clean.filter(df_clean['DIGITAL_CLIENT'] == '1').count())/(df_clean.count())*100
print(f'Porcentaje de los clientes digitales: {Clientes_digitales}')


# ===== PASO 1: AGRUPAR POR CLIENTE Y SUMAR GASTOS =====
# Calcular total de gastos por cliente en tres categorías
df_totals = df_clean.groupBy("CLIENT_ID").agg(
    F.sum("CREDIT_CARD_DRAWINGS_ATM").alias("total_atm"),  # Total retirado en cajeros
    F.sum("CREDIT_CARD_DRAWINGS_POS").alias("total_pos"),  # Total gastado con tarjeta (POS)
    F.sum("CREDIT_CARD_DRAWINGS").alias("total_gastos")  # Total de gastos
)

# ===== PASO 2: CALCULAR PORCENTAJES DE CADA TIPO DE GASTO =====
# Dividir cada gasto entre el total para obtener porcentajes
df_percentages = df_totals.withColumn(
    "pct_atm",  # Porcentaje de gastos en cajero
    (F.col("total_atm") / F.col("total_gastos")) * 100
).withColumn(
    "pct_pos",  # Porcentaje de gastos con tarjeta (POS)
    (F.col("total_pos") / F.col("total_gastos")) * 100
).fillna(0)  # Reemplazar divisiones por cero con 0

# ===== PASO 3: SEGMENTAR CLIENTES POR COMPORTAMIENTO =====
# Clasificar clientes según su patrón de gasto
df_segmented = df_percentages.withColumn(
    "segmento",  # Nombre del segmento
    F.when(F.col("pct_atm") > 70, "Efectivo-dependiente")  # Si usa >70% en ATM
    .when(F.col("pct_pos") > 70, "Cashless")  # Si usa >70% con tarjeta (sin efectivo)
    .otherwise("Mixto")  # Usa tanto efectivo como tarjeta
)

# Mostrar primeros registros
df_segmented.show()

# Contar cuántos clientes hay en cada segmento
(df_segmented.groupBy(
    'segmento').count().orderBy(F.desc('count'))
).show(10)

+--------------+------+
|DIGITAL_CLIENT| count|
+--------------+------+
|           0.0|153832|
|           1.0|  9145|
+--------------+------+

Porcentaje de los clientes digitales: 5.6112212152635035
+------------+---------+---------+------------+-------+-------+--------------------+
|   CLIENT_ID|total_atm|total_pos|total_gastos|pct_atm|pct_pos|            segmento|
+------------+---------+---------+------------+-------+-------+--------------------+
|ES182100018E|      0.0|      0.0|         0.0|    0.0|    0.0|               Mixto|
|ES182100023A|      0.0|      0.0|         0.0|    0.0|    0.0|               Mixto|
|ES182100084M|      0.0|      0.0|         0.0|    0.0|    0.0|               Mixto|
|ES182100087X|      0.0|      0.0|         0.0|    0.0|    0.0|               Mixto|
|ES182100108T|      0.0|      0.0|         0.0|    0.0|    0.0|               Mixto|
|ES182100115E|    540.0|      0.0|       540.0|  100.0|    0.0|Efectivo-dependiente|
|ES182100162C|      0.0|      0.0

# Valor Económico

VEMOS LOS INGRESOS DE UNA PERSONA EN COMPARACION CON LOS PAGOS REALIZADOS A FINPLUS, SI GANA MUCHO Y PAGA POCO ES PORQUE PRINCIPALMENTE TIENE SU DINERO EN OTRO BANCO

In [ ]:
# ===== ANÁLISIS DE SHARE OF WALLET (PORCENTAJE DEL INGRESO) =====
# Objetivo: Determinar qué porcentaje del ingreso total gasta el cliente en el banco
# Clasificar clientes según su potencial de crecimiento

# ===== PASO 1: CALCULAR PAGOS TOTALES AL BANCO =====
# Suma de cuotas de préstamos más los pagos de tarjeta de crédito
df_sow_analysis = df_clean.withColumn(
    "TOTAL_PAYMENTS_TO_BANK",  # Total pagado al banco
    F.col("INSTALLMENT") + F.col('CREDIT_CARD_PAYMENT')
).withColumn(
    "SHARE_OF_WALLET",  # Share of Wallet (porcentaje del ingreso)
    # Si el cliente declara ingresos, calcular ratio; si no, asumir 0
    F.when(F.col("TOTAL_INCOME") > 0, 
           F.col("TOTAL_PAYMENTS_TO_BANK") / F.col("TOTAL_INCOME")
    ).otherwise(0)  # Evitar división por cero
)

# ===== PASO 2: CLASIFICAR ESTRATEGIA COMERCIAL =====
# Según el Share of Wallet, definir la mejor estrategia para cada cliente
df_final_strategy = df_sow_analysis.withColumn(
    "ESTRATEGIA_COMERCIAL",  # Estrategia recomendada
    F.when(F.col("SHARE_OF_WALLET") <= 0.10, "Ataque (Traer Nómina/Hipoteca)")  # Bajo compromiso: ofrecerle más productos
     .when(F.col("SHARE_OF_WALLET").between(0.10, 0.40), "Crecimiento (Cross-Sell)")  # Compromiso moderado: vender productos relacionados
     .when(F.col("SHARE_OF_WALLET").between(0.40, 0.60), "Fidelización (Blindaje)")  # Alto compromiso: proteger la relación
     .when(F.col("SHARE_OF_WALLET") > 0.60, "Alerta Riesgo (Posible Impago)")  # Muy alto: cliente sobreendeudado, riesgo
     .otherwise("Revisar Datos")  # Datos inconsistentes
)

# ===== PASO 3: MOSTRAR ANÁLISIS FINAL =====
# Mostrar 10 registros con ID cliente, ingresos, pagos, ratio y estrategia
df_final_strategy.select(
    "CLIENT_ID", "TOTAL_INCOME", "TOTAL_PAYMENTS_TO_BANK", "SHARE_OF_WALLET", "ESTRATEGIA_COMERCIAL"
).show(10)

(df_final_strategy.groupBy(
    'ESTRATEGIA_COMERCIAL').count().orderBy(F.desc('count'))
).show(10)

+------------+------------+----------------------+-------------------+--------------------+
|   CLIENT_ID|TOTAL_INCOME|TOTAL_PAYMENTS_TO_BANK|    SHARE_OF_WALLET|ESTRATEGIA_COMERCIAL|
+------------+------------+----------------------+-------------------+--------------------+
|ES182411319L|      1350.0|                276.75|              0.205|Crecimiento (Cros...|
|ES182116369S|       918.0|                261.31|0.28465141346252043|Crecimiento (Cros...|
|ES182154395P|      1350.0|                298.24|0.22091851128472223|Crecimiento (Cros...|
|ES182369450D|      1620.0|                386.37| 0.2384999969859182|Crecimiento (Cros...|
|ES182116792X|      2430.0|                245.13|0.10087654521926441|Crecimiento (Cros...|
|ES182302152P|      4860.0|                 810.0|0.16666666666666666|Crecimiento (Cros...|
|ES182354446X|      1620.0|                 466.4|  0.287901230800299|Crecimiento (Cros...|
|ES182390546C|      2430.0|                 243.0|                0.1|Ataque (Tr

CUANTO CREDITO CONSIGUE EL CLIENTE RESPECTO DEL QUE HA PEDIDO

In [107]:
# ===== RATIO DE CONVERSIÓN DE CRÉDITO =====
# Medir qué porcentaje de crédito solicitado fue realmente otorgado
# Indica la "efectividad" del proceso de aprobación

df_clean.select(
    F.col('LOAN_APPLICATION_AMOUNT_SUM'),  # Total solicitado
    F.col('LOAN_CREDIT_GRANTED_SUM')  # Total concedido
).withColumn(
    'CREDIT_GARANTED_SCORE',  # Score de concesión = Concedido / Solicitado
    F.col('LOAN_CREDIT_GRANTED_SUM') / F.col('LOAN_APPLICATION_AMOUNT_SUM')
).show()


+---------------------------+-----------------------+---------------------+
|LOAN_APPLICATION_AMOUNT_SUM|LOAN_CREDIT_GRANTED_SUM|CREDIT_GARANTED_SCORE|
+---------------------------+-----------------------+---------------------+
|                    4210.81|                4248.18|    1.008874804768558|
|                     809.95|                 895.48|   1.1055990702792364|
|                    1704.24|                1624.54|    0.953234314633754|
|                    1989.85|                2105.03|    1.057883787785067|
|                        0.0|                 2160.0|                 NULL|
|                        0.0|                    0.0|                 NULL|
|                   40486.45|               43083.79|    1.064153312870597|
|                     2087.1|                2039.58|   0.9772314985491466|
|                     3501.9|                3313.71|   0.9462606166211952|
|                    2291.27|                2157.25|   0.9415084130683699|
|           

# Interacción y Fidelidad

MEDIMOS QUE TAN VINCULADO ESTA EL CLIENTE CON NUESTRO BANCO

In [ ]:
# ===== ANÁLISIS DE VINCULACIÓN CON EL BANCO =====
# Objetivo: Medir la lealtad del cliente basada en número de productos contratados
# Clientes con más productos están más vinculados y tienen menor riesgo de fuga

# ===== PASO 1: CALCULAR PROMEDIO Y MÁXIMO DE PRODUCTOS =====
# Ver cuál es el número típico y el máximo de productos que tiene un cliente
df_clean.select(F.mean("NUMBER_OF_PRODUCTS"), F.max("NUMBER_OF_PRODUCTS")).show()

# ===== PASO 2: CLASIFICAR POR NIVEL DE VINCULACIÓN =====
df_cross_sell = df_clean.withColumn(
    "VINCULACION",  # Nivel de vinculación
    F.when(F.col("NUMBER_OF_PRODUCTS") == 1, "Baja (Riesgo Fuga)")  # 1 solo producto: alto riesgo
     .when(F.col("NUMBER_OF_PRODUCTS").between(2, 3), "Media")  # 2-3 productos: vinculación media
     .when(F.col("NUMBER_OF_PRODUCTS") >= 4, "Alta (Fidelizado)")  # 4+ productos: cliente fidelizado
     .otherwise("Desconocido")
)

# Contar cuántos clientes en cada categoría
df_cross_sell.groupBy("VINCULACION").count().show()


+-----------------------+-----------------------+
|avg(NUMBER_OF_PRODUCTS)|max(NUMBER_OF_PRODUCTS)|
+-----------------------+-----------------------+
|      1.644523460365573|                   20.5|
+-----------------------+-----------------------+



CALCULAMOS EL RATIO DE RECHAZO DE LOS CLIENTES BASANDONOS EN LAS SOLICITUDES RECHAZADAS, ANULADAS, ACEPTADAS Y AUTORIZADAS DE LOS CLIENTES

In [ ]:
# ===== ANÁLISIS DE RECHAZO DEL CLIENTE =====
# Objetivo: Identificar clientes que frecuentemente rechazan ofertas aprobadas

# ===== PASO 1: CALCULAR TOTAL DE SOLICITUDES HISTÓRICAS =====
df_rechazo = df_clean.withColumn(
    "TOTAL_SOLICITUDES_HISTORICAS",  # Suma de todas las solicitudes
    F.col("NUM_STATUS_ANNULLED") +  # Solicitudes anuladas
    F.col("NUM_STATUS_AUTHORIZED") +  # Solicitudes autorizadas
    F.col("NUM_STATUS_DENIED") +  # Solicitudes rechazadas
    F.col("NUM_STATUS_NOT_USED")  # Solicitudes no utilizadas (RECHAZADAS POR EL CLIENTE)
).withColumn(
    "RATIO_VITRINEO",  # Ratio de "escaparate" (rechaza ofertas aprobadas)
    # Porcentaje de veces que el cliente rechazó una oferta aprobada
    F.when(F.col("TOTAL_SOLICITUDES_HISTORICAS") > 0,
           F.col("NUM_STATUS_NOT_USED") / F.col("TOTAL_SOLICITUDES_HISTORICAS")
    ).otherwise(0)
)

# ===== PASO 2: IDENTIFICAR CLIENTES CON ALTO RATIO DE RECHAZO =====
# Filtrar clientes que rechazan más del 50% de las ofertas aprobadas
# Estos clientes son "escurridizos" - les ofreces crédito pero lo rechaza frecuentemente
df_rechazo.filter(F.col("RATIO_VITRINEO") > 0.5).select("CLIENT_ID", "RATIO_VITRINEO").show(5)

# Mostrar tabla reducida para KPI
# 1. Crear una columna binaria (1/0) para la clasificación de rechazo
df_rechazo_clasificado = df_rechazo.withColumn(
    "ALTO_RECHAZO_FLAG",
    F.when(F.col("RATIO_VITRINEO") > 0.5, 1) # 1 si rechaza más del 50%
    .otherwise(0)                          # 0 si rechaza 50% o menos
)

# 2. Agrupar por el numero anterior y contar el total de clientes en cada grupo
df_conteo_rechazo = df_rechazo_clasificado.groupBy("ALTO_RECHAZO_FLAG").agg(
    F.count("CLIENT_ID").alias("TOTAL_CLIENTES")
)

# 3. Mostrar la tabla final
print("\nConteo de Clientes por Ratio de Rechazo (>50%)")
df_conteo_rechazo.orderBy(F.col("ALTO_RECHAZO_FLAG").desc()).show()


+------------+------------------+
|   CLIENT_ID|    RATIO_VITRINEO|
+------------+------------------+
|ES182201729A|0.6666666666666666|
|ES182241009M|               0.6|
|ES182287694L|              0.75|
|ES182253955P|0.6666666666666666|
|ES182158899Q|0.5454545454545454|
+------------+------------------+
only showing top 5 rows


--- Conteo de Clientes por Ratio de Rechazo (>50%) ---


COMPARACION DE LO QUE EL BANCO CREE QUE SON TRANSACCIONES DIGITALES FRENTE A LAS QUE REALMENTE LO SON PARA ANALIZAR DESAJUSTES

In [49]:
# ===== MATRIZ DE CONFUSIÓN DE DIGITALIZACIÓN =====
# Comparar lo que el banco CREE que es digital con lo que REALMENTE hace

# ===== PASO 1: CALCULAR RATIO DE USO DIGITAL =====
# Dividir gastos en POS (sin efectivo) entre gastos totales
df_digital_kpi = df_clean.withColumn(
    "RATIO_USO_DIGITAL",  # Porcentaje de transacciones sin efectivo
    F.when(F.col("CREDIT_CARD_DRAWINGS") > 0,  # Si hay transacciones
           F.col("CREDIT_CARD_DRAWINGS_POS") / F.col("CREDIT_CARD_DRAWINGS")  # % POS / total
    ).otherwise(0)  # Si no hay transacciones, asumir 0
)

# ===== PASO 2: CREAR MATRIZ DE CONFUSIÓN =====
# Comparar DIGITAL_CLIENT (lo que el banco cree) vs RATIO_USO_DIGITAL (realidad)
# Esto identifica desajustes en la clasificación
df_digital_kpi.withColumn(
    "PERFIL_REAL",  # Clasificación real basada en comportamiento
    F.when((F.col("DIGITAL_CLIENT") == 1) & (F.col("RATIO_USO_DIGITAL") > 0.5), "Digital Puro")  # El banco tiene razón
     .when((F.col("DIGITAL_CLIENT") == 1) & (F.col("RATIO_USO_DIGITAL") <= 0.5), "Digital de Fachada (Usa Cash)")  # Banco error: lo clasifica digital pero usa efectivo
     .when((F.col("DIGITAL_CLIENT") == 0) & (F.col("RATIO_USO_DIGITAL") > 0.5), "Digital Potencial (Sin App)")  # Banco error: lo clasifica analógico pero usa tarjeta
     .otherwise("Analógico")  # El banco tiene razón: usa efectivo
).groupBy("PERFIL_REAL").count().show()


+--------------------+------+
|         PERFIL_REAL| count|
+--------------------+------+
|        Digital Puro|   587|
|Digital Potencial...|  4804|
|           Analógico|149028|
|Digital de Fachad...|  8558|
+--------------------+------+



VEMOS CUAN PRINCIPAL ES EL BANCO PARA EL CLIENTE COMPARANDO EL SALDO DE LA TARJETA CON SU LIMITE Y SU FRECUENCIA DE USO

In [90]:
# ===== ANÁLISIS DE FIDELIDAD POR TASA DE UTILIZACIÓN DE TARJETA (CCU) =====
# Objetivo: Segmentar clientes según cuánto uso hacen del crédito disponible (indica fidelidad/riesgo)

# ===== PASO 1. Calcular la Tasa de Utilización (CCU) =====
# CCU_RATE = Saldo usado / Límite de crédito
# Se protege la división por cero verificando que CREDIT_CARD_LIMIT existe y sea > 0
df_ccu = df_clean.withColumn(
    "CCU_RATE",
    # Cálculo: Saldo usado / Límite de Crédito. Se maneja la división por cero.
    F.when(F.col("CREDIT_CARD_LIMIT").cast("double").isNotNull() & (F.col("CREDIT_CARD_LIMIT") > 0), 
         F.col("CREDICT_CARD_BALANCE") / F.col("CREDIT_CARD_LIMIT")
    ).otherwise(0)
)

# ===== PASO 2. Segmentar por Intensidad de Uso ======
# Definimos tres segmentos basados en CCU_RATE:
# - < 0.10: Dormido / Bajo Uso -> potencial campaña de activación
# - > 0.90: Ahogado / Alto Uso -> posible riesgo o candidato a up-sell (tarjeta casi al límite)
# - Resto: Usuario Principal / Uso Normal
df_fidelidad_final = df_ccu.withColumn(
    "INTENSIDAD_USO_SEGMENTO",
    F.when(F.col("CCU_RATE") < 0.10, "1. Dormido / Bajo Uso (Camp. Activación)")
    .when(F.col("CCU_RATE") > 0.90, "3. Ahogado / Alto Uso (Riesgo o Up-sell)")
    .otherwise("2. Usuario Principal / Uso Normal") 
)

# ===== PASO 3. Análisis de Resultados  =====
# Agregamos indicadores por segmento: número de clientes y frecuencia promedio de uso (NUMBER_DRAWINGS)
# TOTAL_CLIENTES: tamaño del segmento
# PROMEDIO_FRECUENCIA_USO: promedio de transacciones (indicador de engagement)
df_segment_counts = df_fidelidad_final.groupBy("INTENSIDAD_USO_SEGMENTO").agg(
    F.count("CLIENT_ID").alias("TOTAL_CLIENTES"),
    F.avg("NUMBER_DRAWINGS").alias("PROMEDIO_FRECUENCIA_USO") 
)

# Calcular el Porcentaje de cada segmento:
# Paso 3a: Calcular el total de clientes global (suma de TOTAL_CLIENTES)
total_clientes_global_df = df_segment_counts.agg(F.sum("TOTAL_CLIENTES").alias("TOTAL_GLOBAL"))

# Ahora unimos para calcular el porcentaje por segmento respecto al total global
df_resumen_con_porcentaje = df_segment_counts.crossJoin(total_clientes_global_df).withColumn(
    "PORCENTAJE",
    (F.col("TOTAL_CLIENTES") / F.col("TOTAL_GLOBAL")) * 100
)

# Mostrar la tabla de resultados final (ordenada por tamaño de segmento)
print("Segmentación de Clientes por Intensidad de Uso (Fidelidad)")
df_resumen_con_porcentaje.orderBy(F.col("TOTAL_CLIENTES").desc()).show(10,0)

# ===== PASO 4. Preparar datos para impresión y reporting =====
# Extraemos los porcentajes a una lista/local para poder imprimir mensajes resumidos
datos_impresion = df_resumen_con_porcentaje.select("INTENSIDAD_USO_SEGMENTO", "PORCENTAJE").collect()

# Convertir a un diccionario de Python para fácil acceso
porcentajes = {row["INTENSIDAD_USO_SEGMENTO"]: row["PORCENTAJE"] for row in datos_impresion}

# Definiciones de segmentos (variables para acceder al diccionario)
BAJO = "1. Dormido / Bajo Uso (Camp. Activación)"
ALTO = "3. Ahogado / Alto Uso (Riesgo o Up-sell)"
NORMAL = "2. Usuario Principal / Uso Normal"

# Imprimir porcentajes con formato
print(f'Porcentaje de clientes con bajo uso: {porcentajes.get(BAJO, 0):.2f}%')
print(f'Porcentaje de clientes con alto uso: {porcentajes.get(ALTO, 0):.2f}%')
print(f'Porcentaje de clientes con uso normal: {porcentajes.get(NORMAL, 0):.2f}%')

Segmentación de Clientes por Intensidad de Uso (Fidelidad)
+----------------------------------------+--------------+-----------------------+------------+------------------+
|INTENSIDAD_USO_SEGMENTO                 |TOTAL_CLIENTES|PROMEDIO_FRECUENCIA_USO|TOTAL_GLOBAL|PORCENTAJE        |
+----------------------------------------+--------------+-----------------------+------------+------------------+
|1. Dormido / Bajo Uso (Camp. Activación)|147946        |0.04188690468143782    |162977      |90.77722623437663 |
|3. Ahogado / Alto Uso (Riesgo o Up-sell)|8870          |3.047914317925592      |162977      |5.442485749522938 |
|2. Usuario Principal / Uso Normal       |6161          |4.188768057133582      |162977      |3.7802880161004313|
+----------------------------------------+--------------+-----------------------+------------+------------------+

Porcentaje de clientes con bajo uso: 90.78%
Porcentaje de clientes con alto uso: 5.44%
Porcentaje de clientes con uso normal: 3.78%


# Riesgo Potencial

VER CLIENTES QUE VIVEN POR ENCIMA DE SUS POSIBILIDADES, CALCULANDO EL RATIO DE ENDEUDAMIENTO VIENDO SI EL CLIENTE DEBE EN CUOTAS MAS TARJETA MAS DEL 45% DE LO QUE GANA

In [ ]:
# ===== CALCULAR RATIO DE DEUDA  =====
# Rellenar nulos de saldo de tarjeta con 0
df_clean = df_clean.na.fill({'CREDICT_CARD_BALANCE':0.0})

# Calcular Debt Ratio = (Cuota + (Saldo_Tarjeta * 5%)) / Ingreso Total * 100
# La cuota mensual más la parte del saldo de la tarjeta dividido entre el ingreso
df = df_clean.select(
    F.col('INSTALLMENT'),  # Cuota/instalación mensual
    F.col('CREDICT_CARD_BALANCE'),  # Saldo de tarjeta de crédito
    F.col('TOTAL_INCOME'),  # Ingreso total
    # Fórmula: (cuota + 5% del saldo) / ingreso * 100
    (((F.col('INSTALLMENT') + (F.col('CREDICT_CARD_BALANCE') * 0.05)) / F.col('TOTAL_INCOME')) * 100).alias('DEBT_RATIO')
).withColumn(
    "Riesgo",  # Clasificar el riesgo basado en el ratio
    F.when(F.col("DEBT_RATIO") > 45, "Alto Riesgo")  # Si ratio > 45%, riesgo alto
    .otherwise("Bajo Riesgo")  # Si ratio <= 45%, riesgo bajo
)

# Mostrar resultados
df.show()

# Contar cuántos clientes en cada categoría de riesgo
(df.groupBy(
    'Riesgo').count().orderBy(F.desc('count'))
).show(10)

+-----------+--------------------+------------+------------------+-----------+
|INSTALLMENT|CREDICT_CARD_BALANCE|TOTAL_INCOME|        DEBT_RATIO|     Riesgo|
+-----------+--------------------+------------+------------------+-----------+
|     276.75|                 0.0|      1350.0|              20.5|Bajo Riesgo|
|     261.31|                 0.0|       918.0|28.465141346252043|Bajo Riesgo|
|     298.24|                 0.0|      1350.0| 22.09185112847222|Bajo Riesgo|
|     386.37|                 0.0|      1620.0| 23.84999969859182|Bajo Riesgo|
|      242.3|                 0.0|      2430.0| 9.971193541224602|Bajo Riesgo|
|      810.0|                 0.0|      4860.0|16.666666666666664|Bajo Riesgo|
|      466.4|                 0.0|      1620.0|  28.7901230800299|Bajo Riesgo|
|      243.0|                 0.0|      2430.0|              10.0|Bajo Riesgo|
|     455.38|                 0.0|      1080.0|42.164815266927086|Bajo Riesgo|
|     315.79|                 0.0|      4320.0| 7.30

ANALIZAR LA CANTIDAD DE SOLICITUDES RECHAZADAS RESPECTO A LAS TOTALES DE LOS CLIENTES PARA VER SI HAY ALTO RIESGO DE DESCONTENTO Y ABANDONO

In [94]:
# ===== ANÁLISIS DE RIESGO POR TASA DE RECHAZO =====
# Calcular el porcentaje de solicitudes que fueron rechazadas vs aprobadas
# Este es un indicador de riesgo crediticio

df = df_clean.select(
    F.col('NUM_STATUS_DENIED'),  # Solicitudes rechazadas
    F.col('NUM_STATUS_AUTHORIZED'),  # Solicitudes aprobadas
    # Calcular porcentaje de rechazo
    F.when(
        (F.col('NUM_STATUS_AUTHORIZED') + F.col('NUM_STATUS_DENIED')) == 0,  # Si no hay solicitudes
        F.lit(None)  # Valor nulo (no se puede calcular)
    ).otherwise(
        # Porcentaje = (Rechazadas / (Aprobadas + Rechazadas)) * 100
        ((F.col('NUM_STATUS_DENIED') / (F.col('NUM_STATUS_AUTHORIZED') + F.col('NUM_STATUS_DENIED'))) * 100)
    ).alias('PCT_RECHAZO')
).withColumn(
    "Riesgo",  # Clasificación de riesgo
    F.when(F.col("PCT_RECHAZO").isNull(), "Indefinido")  # Sin datos de solicitudes
    .when(F.col("PCT_RECHAZO") > 60, "Alto riesgo")  # Más del 60% rechazado = riesgo alto
    .otherwise("Bajo Riesgo")  # 60% o menos rechazado = bajo riesgo
)
df.show()

# Contar cuántos clientes en cada categoría de riesgo
(df.groupBy(
    'Riesgo').count().orderBy(F.desc('count'))
).show(10)

+-----------------+---------------------+----------------+-----------+
|NUM_STATUS_DENIED|NUM_STATUS_AUTHORIZED|     PCT_RECHAZO|     Riesgo|
+-----------------+---------------------+----------------+-----------+
|              0.0|                  5.0|             0.0|Bajo Riesgo|
|              0.0|                  1.0|             0.0|Bajo Riesgo|
|              0.0|                  3.0|             0.0|Bajo Riesgo|
|              0.0|                  2.0|             0.0|Bajo Riesgo|
|              0.0|                  1.0|             0.0|Bajo Riesgo|
|              0.0|                  0.0|            NULL| Indefinido|
|              0.0|                 10.0|             0.0|Bajo Riesgo|
|              0.0|                  1.0|             0.0|Bajo Riesgo|
|              0.0|                  2.0|             0.0|Bajo Riesgo|
|              0.0|                  2.0|             0.0|Bajo Riesgo|
|              0.0|                  2.0|             0.0|Bajo Riesgo|
|     

DETECTAMOS SI SE RECHAZAN MAS SOLICITUDES EN ZONAS GEOGRÁFICAS CONCRETAS

In [56]:
# ===== ANÁLISIS URBANO VS RURAL =====

# ===== PASO 1: CLASIFICAR CLIENTES POR ZONA GEOGRÁFICA =====
# Usar REGION_SCORE para distinguir zona urbana (score alto) vs rural (score bajo)
df = df_clean.withColumn(
    "ZONA",  # Nueva columna con clasificación
    F.when(F.col("REGION_SCORE") >= 3, "Urbana")  # Score >= 3: urbana
     .otherwise("Rural")  # Score < 3: rural
)

# ===== PASO 2: AGRUPAR DATOS POR ZONA =====
# Calcular estadísticas para cada zona
df_analisis = df.groupBy("ZONA").agg(
    F.count("CLIENT_ID").alias("Num_Clientes"),  # Cantidad de clientes
    F.round(F.avg("TOTAL_INCOME"), 0).alias("Sueldo_Promedio"),  # Ingreso promedio
    F.sum("NUM_STATUS_DENIED").alias("Total_Rechazos")  # Total de solicitudes rechazadas
)

# ===== PASO 3: CALCULAR ÍNDICE DE RECHAZO =====
# Porcentaje = (Total de rechazos / Total de clientes) * 100
df_final = df_analisis.withColumn(
    "Indice_Rechazo",  # Porcentaje de rechazo
    F.round((F.col("Total_Rechazos") / F.col("Num_Clientes")) * 100, 2)
)

# Mostrar resultado
df_final.show()

+-----+------------+---------------+--------------+--------------+
| ZONA|Num_Clientes|Sueldo_Promedio|Total_Rechazos|Indice_Rechazo|
+-----+------------+---------------+--------------+--------------+
|Rural|      162977|         2029.0|      130104.5|         79.83|
+-----+------------+---------------+--------------+--------------+



ANALIZAR SI EL MODELO DE BEHAVIOURAL SOCRING DISCRIMINA BIEN ENTRE CLIENTES

In [ ]:
# ===== VALIDACIÓN DEL MODELO: ANÁLISIS DE BEHAVIORAL SCORING POR QUINTILES =====
# Objetivo: Verificar que el modelo de behavioral scoring discrimina correctamente entre clientes
# Metodología: Dividir clientes en 5 grupos iguales (quintiles) y comparar sus métricas de riesgo
# Esperado: clientes con mejor scoring deben tener menos rechazos y anulaciones

# ===== PASO 1: CREAR VENTANA DE ORDENAMIENTO =====
# Crear una partición ordenada por behavioral scoring (de menor a mayor)
# Esta ventana se usará para calcular percentiles usando ntile
window_spec = Window.orderBy("BEHAVIORAL_SCORING")

# ===== PASO 2: CALCULAR QUINTILES Y ASIGNAR ETIQUETAS =====
# ntile(5): Divide la población en 5 grupos iguales (20% cada uno)
# GRUPO_SCORE: número del quintil (1=peor, 5=mejor)
df_validacion = df_clean.withColumn(
    "GRUPO_SCORE",  # Número de quintil (1 al 5)
    F.ntile(5).over(window_spec)  # Calcular quintil basado en behavioral score ordenado
).withColumn(
    "ETIQUETA_GRUPO",  # Nombre descriptivo del grupo
    F.when(F.col("GRUPO_SCORE") == 1, "1. Muy Bajo (Riesgoso)")  # Quintil 1: peor 20%
     .when(F.col("GRUPO_SCORE") == 5, "5. Muy Alto (VIP)")  # Quintil 5: mejor 20%
     .otherwise("Media")  # Quintiles 2, 3, 4: nivel intermedio
)

# ===== PASO 3: AGREGACIÓN POR QUINTIL =====
# Para cada grupo de behavioral score, calculamos métricas que validan el modelo:
# - SCORE_PROMEDIO: promedio del behavioral score del grupo (debe crecer de Q1 a Q5)
# - PROMEDIO_RECHAZOS: media de solicitudes rechazadas (debe decrecer en mejores grupos)
# - PROMEDIO_ANULACIONES: media de solicitudes anuladas (indicador de compromiso)
# - VOLUMEN_CLIENTES: número de clientes en cada quintil
df_test_modelo = df_validacion.groupBy("ETIQUETA_GRUPO").agg(
    F.avg("BEHAVIORAL_SCORING").alias("SCORE_PROMEDIO"),  # Score promedio del grupo
    F.avg("NUM_STATUS_DENIED").alias("PROMEDIO_RECHAZOS"),  # Rechazos promedio
    F.avg("NUM_STATUS_ANNULLED").alias("PROMEDIO_ANULACIONES"),  # Anulaciones promedio
    F.count("CLIENT_ID").alias("VOLUMEN_CLIENTES")  # Tamaño del grupo
).orderBy("ETIQUETA_GRUPO")  # Ordenar por quintil para ver evolución clara

# ===== PASO 4: MOSTRAR RESULTADOS =====
# truncate=False: mostrar columnas completas sin cortar valores
# Esperamos ver una tendencia clara: peor score → más rechazos; mejor score → menos rechazos
df_test_modelo.show(truncate=False)


+----------------------+-------------------+------------------+--------------------+----------------+
|ETIQUETA_GRUPO        |SCORE_PROMEDIO     |PROMEDIO_RECHAZOS |PROMEDIO_ANULACIONES|VOLUMEN_CLIENTES|
+----------------------+-------------------+------------------+--------------------+----------------+
|1. Muy Bajo (Riesgoso)|0.24131811189176708|1.3363449503006504|1.0555436249846606  |32596           |
|5. Muy Alto (VIP)     |0.7412870344473442 |0.4635987114588127|0.6609909495321369  |32595           |
|Media                 |0.5238026681981693 |0.7305135704497576|0.8300165667887018  |97786           |
+----------------------+-------------------+------------------+--------------------+----------------+



COMPROBAR SI LOS CLIENTES QUE TIENEN TRABAJOS ESTABLES SON BUENOS PAGADORES, OBSERVANDO LA ANTIGUEDAD LABORAL DE LOS CLIENTES Y OBSERVANDO SUS SCORES

In [ ]:
# ===== ANÁLISIS DE ESTABILIDAD LABORAL Y SCORING =====

# ===== PASO 1: CREAR GRUPOS DE ANTIGÜEDAD LABORAL =====
# Clasificar clientes según cuántos años llevan en su trabajo
df_seniority_grouped = df_clean.select('CLIENT_ID', 'JOB_SENIORITY', 'PROACTIVE_SCORING').withColumn(
    "SENIORITY_GROUP",  # Nueva columna con la categoría
    F.when(F.col("JOB_SENIORITY") <= 2, "Baja Seniority (0-2 años)")  # Menos de 2 años
    .when((F.col("JOB_SENIORITY") > 2) & (F.col("JOB_SENIORITY") <= 5), "Media Seniority (2-5 años)")  # 2-5 años
    .otherwise("Alta Seniority (> 5 años)")  # Más de 5 años
)

# ===== PASO 2: AGRUPAR POR SENIORITY Y CALCULAR PROMEDIO DE SCORE =====
# Para cada grupo de antigüedad, calcular:
# - Promedio de behavioral score (indicador de confiabilidad)
# - Número de clientes en ese grupo
df_analisis_estabilidad = df_seniority_grouped.groupBy("SENIORITY_GROUP").agg(
    F.avg("PROACTIVE_SCORING").alias("PROMEDIO_SCORE"),  # Score promedio
    F.count("CLIENT_ID").alias("NUM_CLIENTES_GRUPO")  # Cantidad de clientes
)

# ===== PASO 3: MOSTRAR RESULTADOS ORDENADOS =====
# Ordenar por score promedio (clientes más estables primero)
df_analisis_estabilidad.orderBy(F.col("PROMEDIO_SCORE").desc()).show(10,0)

# Oportunidades Comerciales

IDENTIFICAMOS A QUIEN LE PODEMOS DAR UN PRÉSTAMO PRECONCEDIDO, OBSERVAMOS AQUELLOS A LOS QUE YA SE LES RECHAZARON PRÉSTAMOS Y VEMOS SI SU SITUACIÓN HA MEJORADO PARA DARLES UN PRÉSTAMO PRECONCEDIDO

In [ ]:
# ===== IDENTIFICAR CLIENTES CON ALTO POTENCIAL PARA CROSS-SELL =====
# Objetivo: Encontrar clientes que pueden recibir ofertas de productos adicionales

# ===== PASO 1: FILTRAR CLIENTES CON SOLICITUDES RECHAZADAS =====
# Seleccionar solo clientes que han tenido solicitudes denegadas
# Estos son clientes interesados pero que fueron rechazados antes
df_cazadores = df_clean.filter(df_clean['NUM_STATUS_DENIED'] > 0)

# ===== PASO 2: DEFINIR CRITERIOS DE FILTRADO =====
# Usamos percentiles para identificar clientes de "calidad" (ingresos y comportamiento buenos)

# Percentil 75 de ingresos (Q3) - clientes con ingresos altos
PROB_INGRESO = 0.75 

# Percentil 60 de behavioral scoring - clientes con buen comportamiento
PROB_SCORE = 0.60

# ===== PASO 3: CALCULAR UMBRALES =====
# Encontrar el valor de ingresos en el percentil 75
# approxQuantile devuelve una lista; [0] extrae el primer valor
umbral_ingreso_alto = df_clean.stat.approxQuantile(
    "TOTAL_INCOME", 
    [PROB_INGRESO],  # Percentiles a calcular (0.75 = 75%)
    0.01  # Precisión (0.01 = 1%)
)[0]

# Encontrar el behavioral score en el percentil 60
umbral_score_alto = df_clean.stat.approxQuantile(
    "BEHAVIORAL_SCORING", 
    [PROB_SCORE],
    0.01
)[0]

# ===== PASO 4: FILTRAR CLIENTES CON POTENCIAL =====
# Mantener solo clientes que cumplen AMBAS condiciones:
# 1. Tienen ingresos >= percentil 75 (clientes ricos)
# 2. Tienen behavioral score >= percentil 60 (buen comportamiento)
df_cazadores_filtrado = df_cazadores.filter(
    (df_cazadores['TOTAL_INCOME'] >= umbral_ingreso_alto) & 
    (df_cazadores['BEHAVIORAL_SCORING'] >= umbral_score_alto)
)

# ===== PASO 5: CALCULAR TASA DE UTILIZACIÓN DE CRÉDITO =====
# Ratio = Saldo Actual / Límite de Crédito
# Valores bajos indican que el cliente puede pedir más crédito
df_cazadores_final = df_cazadores_filtrado.withColumn(
    "CC_UTILIZATION_RATE", 
    F.col("CREDICT_CARD_BALANCE") / F.col("CREDIT_CARD_LIMIT")
)

# ===== PASO 6: CALCULAR PUNTAJE DE OPORTUNIDAD =====
# Score = Behavioral Scoring * (1 - Utilización)
# Clientes con buen comportamiento y bajo uso de crédito son mejores candidatos
df_cazadores_final = df_cazadores_final.withColumn(
    "OPORTUNIDAD_SCORE",
    F.col("BEHAVIORAL_SCORING") * (1 - F.col("CC_UTILIZATION_RATE"))
)

# ===== PASO 7: ORDENAR POR OPORTUNIDAD =====
# Clientes con mayor score de oportunidad primero
df_cazadores_final = df_cazadores_final.orderBy(F.col("OPORTUNIDAD_SCORE").desc())

# Mostrar los 20 mejores candidatos para ofertas de crédito pre-aprobado
df_cazadores_final.show(20)

# Filtrar para reducir la tabla
df_cazadores_final = df_cazadores_final.withColumn(
    "CLASIFICACION_OPORTUNIDAD",
    F.when(F.col("OPORTUNIDAD_SCORE") > 0.70, "1. Buen Candidato (Alta Prioridad)")
    .when((F.col("OPORTUNIDAD_SCORE") <= 0.70) & (F.col("OPORTUNIDAD_SCORE") >= 0.50), "2. Candidato a Mejorar (Media Prioridad)")
    .otherwise("3. Mal Candidato (Baja Prioridad)")
)

# Opcional: Mostrar la distribución de la nueva clasificación
print("\nDistribución de Candidatos por Nivel de Oportunidad")
df_cazadores_final.groupBy("CLASIFICACION_OPORTUNIDAD").count().orderBy(F.col("count").desc()).show(truncate=False)

+------------+----------------------+-----------------+------+------------+--------------+-----------+---------+--------------+--------------------+------------+------------+-------------+--------------+-----------+-----------------+-------+-----------+----------------+-----------------+------------------+---------------------+------------------+----------+--------------+----------+--------------------------+--------+---------------------+------------------------+------------------------+------------------------+---------------------------+---------------------------+---------------------------+-----------------------+-----------------------+-----------------------+----------------------+----------------------+-------------------+---------------------+-----------------+-------------------+----------------+------------------+----------+--------------------+-----------------+------------------------+--------------------+------------------------+--------------------------+----------------

COMPROBAR SI LOS CLIENTES TIENEN EL SEGURO CON NOSOTROS O SI ES POSIBLE QUE LO TENGAN CON OTRO BANCO

In [ ]:
# ===== ANÁLISIS PRELIMINAR: DISTRIBUCIÓN DE COCHES E SEGUROS =====
# Objetivo: Ver la combinación de edad de coche y tenencia de seguro

# Usar el dataframe limpio
df = df_clean

# Agrupar por dos columnas:
# - CAR_AGE: edad del coche (en años)
# - OWN_INSURANCE_CAR: si tiene seguro de auto ('Y' = sí, 'N' = no)
# Contar cuántos clientes hay en cada combinación
# Ordenar de mayor a menor (F.desc('count')): mostrar primero las combinaciones más frecuentes
(df.groupBy('CAR_AGE', 'OWN_INSURANCE_CAR')
    .count()
    .orderBy(F.desc('count'))
).show()


+-------+-----------------+------+
|CAR_AGE|OWN_INSURANCE_CAR| count|
+-------+-----------------+------+
|    0.0|                N|107548|
|    7.0|                Y|  3949|
|    6.0|                Y|  3364|
|    3.0|                Y|  3345|
|    2.0|                Y|  3163|
|    8.0|                Y|  3123|
|    4.0|                Y|  2924|
|    1.0|                Y|  2776|
|    9.0|                Y|  2715|
|   10.0|                Y|  2543|
|   13.0|                Y|  2459|
|   14.0|                Y|  2404|
|   11.0|                Y|  2269|
|   12.0|                Y|  2261|
|    5.0|                Y|  1926|
|   15.0|                Y|  1860|
|   16.0|                Y|  1758|
|   17.0|                Y|  1527|
|   18.0|                Y|  1272|
|   64.0|                Y|  1256|
+-------+-----------------+------+
only showing top 20 rows



IDENTIFICAR CAMBIO, RENOVACIÓN O VENTA DE COCHE DE LOS CLIENTES Y LA PRIORIDAD SEGÚN LA EDAD DEL COCHE

In [ ]:
# ===== IDENTIFICAR CLIENTES PARA OPORTUNIDAD DE RENOVACIÓN DE VEHÍCULOS =====
# Objetivo: Encontrar clientes con coches antiguos que podrían renovar su vehículo
# Estrategia: Enfocarse en clientes que ya tienen seguro (compromiso previo) e ingresos suficientes

# ===== PASO 1: FILTRAR CLIENTES CANDIDATOS =====
# Criterios de selección:
# 1. OWN_INSURANCE_CAR == 'Y': Cliente ya tiene seguro de coche (indica propiedad confirmada)
# 2. CAR_AGE >= 10: El coche tiene 10+ años (antiguo, posible renovación)
# 3. TOTAL_INCOME >= 2500: Ingreso mínimo para poder financiar vehículo nuevo
df_coche = df_clean.filter(
    (F.col("OWN_INSURANCE_CAR") == 'Y') &       
    (F.col("CAR_AGE") >= 10) &                
    (F.col("TOTAL_INCOME") >= 2500)           
)

# Mostrar muestra de clientes seleccionados
df_coche.show()

# ===== PASO 2: CREAR COLUMNA DE PRIORIDAD PARA CALL CENTER =====
# Clasificar clientes según urgencia de renovación basado en edad del vehículo
# El Call Center llamará primero a los "Urgente" (coches muy viejos = más probabilidad de compra)
df_priorizado = df_coche.withColumn(
    "PRIORIDAD_VENTA",  # Nueva columna con nivel de urgencia
    F.when(F.col("CAR_AGE") > 15, "1. Urgente (Chatarra)")  # Coche >15 años: muy viejo, "casi chatarra"
     .when(F.col("CAR_AGE").between(10, 15), "2. Alta (Renovación)")  # Coche 10-15 años: edad de renovación típica
     .otherwise("3. Media")  # Coches en otros rangos
)

# ===== PASO 3: SELECCIONAR COLUMNAS Y ORDENAR POR CAPACIDAD DE COMPRA =====
# Objetivo: Crear lista priorizada para el equipo de ventas
# Mostrar los clientes con mayor ingreso primero (capacidad de pago para vehículo nuevo)
df_recambio_coche = df_priorizado.select(
    F.col("CLIENT_ID"),  # ID único del cliente
    F.col("NAME_PRODUCT_TYPE"),  # Tipo de producto que tiene
    F.col("CAR_AGE"),  # Edad actual del vehículo (en años)
    F.col("TOTAL_INCOME"),  # Ingreso total (indicador de capacidad financiera)
    F.col("PRIORIDAD_VENTA")  # Nivel de urgencia de venta
).orderBy(F.col("TOTAL_INCOME").desc())  # Ordenar descendente: ingresos altos primero

# Mostrar los 10 clientes más atractivos (ingresos altos + prioridad de renovación)
df_recambio_coche.show(10)

# ===== PASO 4: ANÁLISIS DE DISTRIBUCIÓN POR PRIORIDAD =====
# Ver cuántos clientes hay en cada nivel de prioridad
# Esto ayuda a dimensionar la campaña de Call Center
df_recambio_coche.groupBy('PRIORIDAD_VENTA').count().orderBy(F.desc('count')).show()

+------------+----------------------+-----------------+------+------------+--------------+-----------+---------+--------------+--------------------+------------+------------+-------------+--------------+-----------+-----------------+-------+-----------+----------------+-----------------+------------------+---------------------+------------------+----------+--------------+----------+--------------------------+--------+---------------------+------------------------+------------------------+------------------------+---------------------------+---------------------------+---------------------------+-----------------------+-----------------------+-----------------------+----------------------+----------------------+-------------------+---------------------+-----------------+-------------------+----------------+------------------+----------+--------------------+-----------------+------------------------+--------------------+------------------------+--------------------------+----------------

# Almacenamiento

In [ ]:
# ===== GUARDAR DATAFRAME LIMPIO EN CSV =====
# Exportar df_clean a formato CSV (con encabezados)
# Este es el dataset final limpio y listo para análisis
df_clean.write.csv(DATA_PATH + 'client_beh.csv', header=True)